<a href="https://colab.research.google.com/github/Aisyaaswy/Pemrograman_Mesin/blob/main/jobsheet_6/HNSW/praktikum_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 11.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 80.9 MB/s eta 0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520033 sha256=285bcb940aa8eb54c1c108412682ed675b5227b76b00712fb856610293d2c5cd
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=82f8cbcb0ac01ed79a6fd760d201e7c16f86b25473aee714e1e168e6e26e995e
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built annoy hnswlib


In [3]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [7]:
# -------------------------------
# Load dataset
# -------------------------------
df = pd.read_csv('songs_with_attributes_and_lyrics.csv', on_bad_lines='skip', engine='python')
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

In [8]:
# -------------------------------
# Exact Nearest Neighbor (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")


Exact NN done in 7.700 s


In [9]:
# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

Annoy done in 1.656 s


In [10]:
# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

HNSW done in 7.862 s


In [14]:
# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

FAISS IVF done in 3.127 s


In [15]:
# -------------------------------
# Contoh tampilkan top-5 neighbors dari item pertama
# -------------------------------
print("\nTop-5 neighbors for first song:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")


Top-5 neighbors for first song:
Exact NN: [    0  3836 18028  2788 29652]
Annoy:    [0, 3836, 18028, 2788, 29652]
HNSW:     [    0  3836 18028  2788 29652]
FAISS:    [    0  3836 18028  2788 29652]


## **Analisis Tahap Eksperimen (Data Acak)**:
Tahap ini berfokus pada kalibrasi parameter dan evaluasi karakteristik teoretis algoritma
  - Tahap ini berfokus pada kalibrasi parameter dan evaluasi karakteristik teoretis algoritma.

  - Program menggunakan data vektor acak yang dihasilkan melalui pustaka NumPy.

  - Tujuan utamanya adalah mengukur kompromi (trade-off) antara tingkat akurasi (recall) dan waktu komputasi.

  - Program menjalankan iterasi pada parameter spesifik masing-masing algoritma (seperti jumlah pohon pada Annoy atau kedalaman penjelajahan pada FAISS).

  - Output akhir berupa visualisasi grafik yang menunjukkan titik optimal efisiensi dari masing-masing algoritma.

## **Analisis Tahap Implementasi (Dataset Spotify)**
Tahap ini menguji performa mentah algoritma pada lingkungan data nyata.

  - Program memproses dataset lagu nyata dengan mengambil sembilan fitur numerik numerik dominan (seperti tempo, tingkat energi, dan probabilitas tarian).

  - Sistem menerapkan StandardScaler untuk menyamakan skala rentang nilai antar fitur sebelum melakukan perhitungan jarak.

  - Pengujian menggunakan parameter tetap untuk membandingkan kecepatan total waktu eksekusi algoritma Approximate melawan metode tradisional Exact NN.

  - Hasil eksekusi membuktikan FAISS, Annoy, dan HNSW mampu memangkas waktu pencarian secara drastis dengan hasil rekomendasi indeks yang tetap relevan.

## **Analisis Tahap Implementasi (Dataset Spotify)**
Tahap ini menguji performa mentah algoritma pada lingkungan data nyata.

  - Program memproses dataset lagu nyata dengan mengambil sembilan fitur numerik numerik dominan (seperti tempo, tingkat energi, dan probabilitas tarian).

  - Sistem menerapkan StandardScaler untuk menyamakan skala rentang nilai antar fitur sebelum melakukan perhitungan jarak.

  - Pengujian menggunakan parameter tetap untuk membandingkan kecepatan total waktu eksekusi algoritma Approximate melawan metode tradisional Exact NN.

  - Hasil eksekusi membuktikan FAISS, Annoy, dan HNSW mampu memangkas waktu pencarian secara drastis dengan hasil rekomendasi indeks yang tetap relevan.

## **Penyesuaian dan Pemecahan Masalah Kode**
Beberapa modifikasi dilakukan pada kode awal untuk menyesuaikan dengan kondisi lingkungan Google Colab dan struktur dataset nyata.

  - Bypass Data Korup: Penambahan argumen on_bad_lines='skip' pada fungsi pembaca file CSV agar Pandas mengabaikan baris lirik lagu yang memiliki format tanda kutip rusak.

  - Standardisasi Tipe Data: Konversi tipe data hasil normalisasi menjadi float32. Pustaka FAISS dan HNSW akan mengalami kegagalan sistem jika menerima format float64 bawaan dari Pandas.

  - Pembatasan Skala Kueri: Penambahan instruksi X_scaled[:1000] untuk membatasi kueri pencarian hanya pada 1.000 lagu pertama. Langkah ini mencegah kehabisan memori RAM saat sistem menjalankan perhitungan matriks Exact NN secara menyeluruh.

  - Deteksi Dimensi Dinamis: Penggantian variabel angka dimensi statis menjadi X_scaled.shape[1]. Program kini mendeteksi jumlah atribut lagu secara otomatis dari bentuk dataset.

  - Penyesuaian Sintaks Pustaka: Penghapusan nama argumen pada parameter IndexIVFFlat FAISS. Versi pustaka di Google Colab saat ini mengurutkan parameter murni berdasarkan posisi penulisan, bukan nama label argumen.

Implementasi algoritma Approximate Nearest Neighbors mutlak diperlukan untuk membangun sistem rekomendasi atau pencarian pada basis data berskala besar. Metode pencarian konvensional tidak lagi efisien secara waktu dan memori, sehingga transisi ke algoritma berbasis graf atau clustering seperti FAISS dan HNSW memberikan solusi yang praktis dan terukur.